In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from utils import preprocess_data, get_features_and_target
from classification_utils import (
    train_and_compare_classification,
    print_results_table,
    print_confusion_matrix
)
from sklearn.metrics import confusion_matrix

In [2]:
#Подготовка данных
df = preprocess_data('data.xlsx')

# Получаем признаки и бинарную целевую переменную
X, y = get_features_and_target(df, 'IC50, mM',
                                task_type='classification',
                                threshold=None)

# Считаем медиану для информации
median_val = df['IC50, mM'].median()

print(f"\nЦелевая переменная: IC50 > медианы (медиана = {median_val:.3f})")
print(f"Размер X: {X.shape}, размер y: {y.shape}")
print(f"\nБаланс классов:")
print(f"  Класс 0 (IC50 <= медианы): {(y == 0).sum()} ({(y == 0).mean()*100:.1f}%)")
print(f"  Класс 1 (IC50 > медианы):  {(y == 1).sum()} ({(y == 1).mean()*100:.1f}%)")

Загружено: 1001 объектов, 213 столбцов
Удалено константных признаков: 33
Пропуски заполнены медианой каждого столбца
Удалено признаков с корреляцией > 0.95: 33
Итого: 1001 объектов, 147 столбцов (144 признаков)

Целевая переменная: IC50 > медианы (медиана = 46.585)
Размер X: (1001, 144), размер y: (1001,)

Баланс классов:
  Класс 0 (IC50 <= медианы): 501 (50.0%)
  Класс 1 (IC50 > медианы):  500 (50.0%)


In [3]:
#Обучение моделей
results_df, best_models, splits = train_and_compare_classification(
    X, y,
    target_name='IC50 > медианы',
    cv=5,
    test_size=0.3,
    random_state=42
)

X_train, X_test, y_train, y_test = splits

In [4]:
print_results_table(results_df, target_name='IC50 > медианы')

from pathlib import Path; Path('results').mkdir(parents=True, exist_ok=True)
results_df.to_csv('results/results_classification_IC50_median.csv', index=False)

             Model    CV F1  Test Accuracy  Test Precision  Test Recall  Test F1  Test ROC_AUC
  GradientBoosting 0.697924       0.707641        0.670330     0.813333 0.734940      0.780022
      DecisionTree 0.704369       0.691030        0.647668     0.833333 0.728863      0.739801
      RandomForest 0.709423       0.684385        0.653631     0.780000 0.711246      0.772517
               SVC 0.716158       0.684385        0.660819     0.753333 0.704050      0.724260
LogisticRegression 0.686749       0.671096        0.644068     0.760000 0.697248      0.729779
               KNN 0.702694       0.674419        0.668831     0.686667 0.677632      0.706777

Лучшая модель: GradientBoosting (Test F1 = 0.7349)


In [7]:
#Визуализация лучшей модели
# Сравнение моделей по F1-score
plt.figure(figsize=(12, 6))
sorted_df = results_df.sort_values('Test F1', ascending=True)
plt.barh(sorted_df['Model'], sorted_df['Test F1'], color='steelblue', edgecolor='black')
plt.xlabel('F1-score на тестовой выборке')
plt.title('Сравнение моделей классификации по F1-score ')
plt.axvline(x=0.5, color='red', linestyle='--', linewidth=1, label='Случайное угадывание')
plt.legend()
plt.tight_layout()
plt.savefig('results/comparison_classification_IC50_median.png', dpi=100, bbox_inches='tight')
plt.close()

# Подробный анализ лучшей модели
best_model_name = results_df.iloc[0]['Model']
best_model = best_models[best_model_name]

y_pred_test = best_model.predict(X_test)

#Матрица ошибок
print_confusion_matrix(y_test, y_pred_test, model_name=best_model_name)

# Визуализация матрицы ошибок
import seaborn as sns

cm = confusion_matrix(y_test, y_pred_test)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Класс 0', 'Класс 1'],
            yticklabels=['Класс 0', 'Класс 1'])
plt.xlabel('Предсказанный класс')
plt.ylabel('Истинный класс')
plt.title(f'Матрица ошибок: {best_model_name}\n(IC50 > медианы)')
plt.tight_layout()
plt.savefig('results/confusion_matrix_IC50_median.png', dpi=100, bbox_inches='tight')
plt.close()


TN (правильно отрицательные): 91
FP (ложные срабатывания):     60
FN (пропущенные положительные): 28
TP (правильно положительные): 122
              precision    recall  f1-score   support

           0       0.76      0.60      0.67       151
           1       0.67      0.81      0.73       150

    accuracy                           0.71       301
   macro avg       0.72      0.71      0.70       301
weighted avg       0.72      0.71      0.70       301



In [8]:
#Важность признаков
final_model = best_model.named_steps['model']
if hasattr(final_model, 'feature_importances_'):
    print(f"\n Топ-15 важных признаков для {best_model_name} ")
    importances = pd.Series(final_model.feature_importances_, index=X.columns)
    top_features = importances.sort_values(ascending=False).head(15)
    for feat, imp in top_features.items():
        print(f"  {feat:30s}: {imp:.4f}")

    plt.figure(figsize=(10, 6))
    top_features.sort_values().plot(kind='barh', color='coral', edgecolor='black')
    plt.xlabel('Важность признака')
    plt.title(f'Топ-15 признаков для классификации IC50 > медианы\n({best_model_name})')
    plt.tight_layout()
    plt.savefig('results/feature_importance_IC50_median.png', dpi=100, bbox_inches='tight')
    plt.close()


 Топ-15 важных признаков для GradientBoosting 
  VSA_EState8                   : 0.1067
  BCUT2D_MRLOW                  : 0.0718
  NHOHCount                     : 0.0659
  VSA_EState5                   : 0.0391
  MaxAbsPartialCharge           : 0.0363
  FpDensityMorgan3              : 0.0318
  EState_VSA4                   : 0.0305
  SMR_VSA10                     : 0.0294
  BCUT2D_MWLOW                  : 0.0255
  NumSaturatedHeterocycles      : 0.0241
  MolLogP                       : 0.0212
  VSA_EState6                   : 0.0207
  EState_VSA7                   : 0.0194
  TPSA                          : 0.0186
  SlogP_VSA5                    : 0.0148
